# HPD 2 — Extensión (ejercicios evaluables avanzados)

<figure>
<a
href="https://colab.research.google.com/github/Adamychen/m10_quarto/blob/main/notebooks/evaluables/hpd2-extension.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open In Colab</figcaption>
</figure>

> **Peso en la nota:** 7.5 % extra (parte del 30 % de entregas
> prácticas)
>
> **Plazo:** 14 días tras la sesión presencial.
>
> **Requisito previo:** haber completado `hpd2-evaluables.qmd`.
>
> **Entrega:** Notebook `.ipynb` ejecutado. Cada ejercicio especifica
> qué variable debe contener el resultado para la corrección automática.

> **Cómo se corrige**
>
> Cada ejercicio pide que asignes el resultado a una variable con un
> nombre concreto. El script de corrección ejecutará tu notebook e
> inspeccionará esas variables. **Si la variable no existe o tiene un
> tipo incorrecto, el ejercicio se puntúa como 0.**
>
> Para autoevaluarte antes de entregar:
>
> ``` bash
> python scripts/corregir_hpd2.py --extension tu_notebook.ipynb
> ```

In [1]:
!pip install -q langchain langchain-openai langchain-community pandas matplotlib seaborn python-dotenv langgraph chromadb

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io, base64

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool

plt.rcParams["figure.dpi"] = 100

LLM_KEY = os.getenv("LLM_API_KEY")
LLM_URL = "https://llamus.cs.us.es/api/v1"

if not LLM_KEY:
    print("⚠️  Crea .env con LLM_API_KEY=tu_key.")
else:
    llm = ChatOpenAI(model="gemma4:e2b-mlx", base_url=LLM_URL, api_key=LLM_KEY, temperature=0)
    print("✅ LLM configurado.")

df = sns.load_dataset("titanic")

# Herramientas base
@tool
def consulta_pandas(expresion: str) -> str:
    """Ejecuta una expresión de pandas sobre df (Titanic)."""
    try:
        resultado = eval(expresion, {"df": df, "pd": pd, "__builtins__": {}})
        return str(resultado)
    except Exception as e:
        return f"Error: {e}"

@tool
def histograma(columna: str, titulo: str = "") -> str:
    """Genera un histograma de una columna numérica del Titanic."""
    df[columna].hist()
    plt.title(titulo or f"Histograma de {columna}")
    return f"Histograma de {columna} generado"

@tool
def barras(columna: str, agrupar_por: str = "") -> str:
    """Genera un gráfico de barras. agrupar_por: opcional."""
    if agrupar_por:
        df.groupby(agrupar_por)[columna].mean().plot.bar()
    else:
        df[columna].value_counts().plot.bar()
    plt.title(f"Barras de {columna}")
    return f"Barras de {columna} generado"

@tool
def boxplot(columna: str, agrupar_por: str = "") -> str:
    """Genera un boxplot. agrupar_por: opcional."""
    if agrupar_por:
        df.boxplot(columna, by=agrupar_por)
    else:
        df.boxplot(columna)
    return f"Boxplot de {columna} generado"

@tool
def codigo_libre(codigo: str) -> str:
    """
    Ejecuta código matplotlib libre sobre df (Titanic).
    Úsalo para gráficos que ninguna otra tool cubra.

    REGLAS: usa plt.<funcion>(), NUNCA df.plot().
    NO uses plt.show() (se elimina automáticamente).
    NO leas archivos (df ya cargado).
    Variables: df, plt, pd, np.
    """
    codigo = codigo.replace("plt.show()", "")
    try:
        plt.figure()
        exec(codigo, {"plt": plt, "pd": pd, "df": df, "np": np})
        buf = io.BytesIO()
        plt.savefig(buf, format="png", bbox_inches="tight")
        plt.close()
        return f"Gráfico generado ({len(buf.getvalue())} bytes)"
    except Exception as e:
        plt.close()
        return f"Error: {e}. Reintenta con plt.<funcion>() en vez de df.plot()"

SYSTEM_PROMPT = "Eres un analista de datos del Titanic. Usa las herramientas disponibles. Responde en español."

print("Setup listo.")

✅ LLM configurado.
Setup listo.

------------------------------------------------------------------------

## Ejercicio 1 — Agente multi-herramienta (2.5 puntos)

Implementa un agente con **4 o más herramientas** distintas (mínimo:
consulta_pandas, histograma/barras/boxplot, correlacion,
resumen_estadistico). Haz **3 preguntas complejas** que requieran el uso
de al menos 2 herramientas distintas cada una. Guarda el log completo de
ejecución.

| Criterio                                                 | Puntos |
|----------------------------------------------------------|--------|
| 4+ herramientas implementadas con docstrings             | 1.0    |
| 3 preguntas que requieren 2+ tools cada una              | 1.0    |
| Log guardado en `ext1_log` con razonamiento y resultados | 0.5    |

In [3]:
# ─── Implementa 2 herramientas adicionales (mínimo) ───
@tool
def correlacion(col1: str, col2: str) -> str:
    """Calcula la correlación de Pearson entre dos columnas numéricas de df."""
    # TODO: implementar
    pass

@tool
def resumen_estadistico(columna: str) -> str:
    """Devuelve media, mediana, std, min, max de una columna numérica de df."""
    # TODO: implementar
    pass

# ─── Preguntas complejas ───
# 1. pregunta que requiera consulta_pandas + histograma/barras/boxplot

# 3. pregunta que requiera correlacion + resumen_estadistico + alguna visualización

# ─── Resultado esperado por el corrector ───
# ext1_log debe ser una lista de 3 dicts:
# [{"pregunta": str, "tools_usadas": list[str], "respuesta": str, "acierto": bool}, ...]

ext1_log = []  # ← lista de 3 dicts

In [4]:
# ─── Auto-verificación ───
assert isinstance(ext1_log, list), "❌ ext1_log debe ser una lista"
assert len(ext1_log) == 3, f"❌ 3 preguntas esperadas, tienes {len(ext1_log)}"
for i, row in enumerate(ext1_log):
    for k in ("pregunta", "tools_usadas", "respuesta", "acierto"):
        assert k in row, f"❌ Falta '{k}' en fila {i}"
    assert isinstance(row["tools_usadas"], list), f"❌ tools_usadas debe ser lista en fila {i}"
    assert len(row["tools_usadas"]) >= 2, f"❌ Se esperaban ≥2 tools en fila {i}"
print("✅ Ejercicio 1: formato correcto")

------------------------------------------------------------------------

## Ejercicio 2 — Agente con RAG y memoria (2.5 puntos)

Implementa un agente que combine **3 capacidades**: 1. Consultar el
DataFrame Titanic (consulta_pandas) 2. Buscar en un documento de texto
sobre el Titanic (retriever con Chroma) 3. Memoria conversacional
(recordar preguntas anteriores)

Crea un documento de ejemplo con contexto histórico del Titanic (~300
palabras). El agente debe decidir si buscar en el DataFrame (datos) o en
el documento (contexto histórico). Prueba con 2 preguntas de datos y 2
de contexto.

| Criterio                                                   | Puntos |
|------------------------------------------------------------|--------|
| Retriever con Chroma sobre documento del Titanic           | 1.0    |
| Memoria conversacional funcional (3+ turnos)               | 0.75   |
| 4 preguntas evaluadas en `ext2_log` (2 datos + 2 contexto) | 0.75   |

In [5]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document as LCDoc

# ─── Crea el documento histórico ───
# documento = """El RMS Titanic fue un transatlántico británico que naufragó
# en la madrugada del 15 de abril de 1912 durante su viaje inaugural..."""  # (300+ palabras)

# ─── Crea el retriever ───
# 1. Trocea el documento con RecursiveCharacterTextSplitter
# 2. Indexa en Chroma con OpenAIEmbeddings
# 3. Crea retriever con .as_retriever()

# ─── Define una 3ª tool: buscar_contexto(consulta) ───
# Usa el retriever para buscar chunks relevantes y devuelve el texto

# ─── Añade memoria conversacional ───
# Usa una lista de mensajes: historial.append({"role": msg.role, "content": msg.content})
# En create_agent, el historial se pasa como mensajes adicionales en la lista "messages"

# ─── Preguntas de prueba ───
# Datos: "¿Cuál era la edad media de los pasajeros?", "¿Cuántos pasajeros de 1ª clase sobrevivieron?"
# Contexto: "¿Cuándo y dónde naufragó el Titanic?", "¿Cuántos pasajeros llevaba a bordo?"

# ─── Resultado esperado ───
# ext2_log debe ser una lista de 4 dicts:
# [{"pregunta": str, "tipo": "datos"|"contexto", "tool_usada": str, "respuesta": str}, ...]

ext2_log = []  # ← lista de 4 dicts

In [6]:
# ─── Auto-verificación ───
assert isinstance(ext2_log, list), "❌ ext2_log debe ser una lista"
assert len(ext2_log) == 4, f"❌ 4 preguntas esperadas, tienes {len(ext2_log)}"
tipos = [r["tipo"] for r in ext2_log]
assert tipos.count("datos") >= 2, f"❌ Se esperaban ≥2 preguntas de datos, hay {tipos.count('datos')}"
assert tipos.count("contexto") >= 2, f"❌ Se esperaban ≥2 de contexto, hay {tipos.count('contexto')}"
print("✅ Ejercicio 2: formato correcto")

------------------------------------------------------------------------

## Ejercicio 3 — Benchmark de modelos (2.5 puntos)

Ejecuta el mismo agente (consulta_pandas + tools de visualización)
usando **2 LLMs distintos** (el que tengas disponible: gemma4:e2b-mlx +
otro como gpt-3.5-turbo, llama3.2, o cualquier otro vía Llamus US).
Evalúa las mismas 5 preguntas con cada modelo y compara: velocidad,
tokens usados (si la API lo reporta) y aciertos.

| Criterio                               | Puntos |
|----------------------------------------|--------|
| 2 modelos configurados y funcionales   | 1.0    |
| Mismas 5 preguntas ejecutadas en ambos | 1.0    |
| Tabla comparativa en `ext3_tabla`      | 0.5    |

In [7]:
# ─── Configura un segundo LLM ───
# llm2 = ChatOpenAI(model="OTRO_MODELO", base_url=LLM_URL, api_key=LLM_KEY, temperature=0)

# ─── 5 preguntas de benchmark ───
# Mismas para ambos modelos

# ─── Ejecuta y mide ───
# Para cada modelo y cada pregunta, anota:
# - Tiempo de ejecución
# - ¿Respuesta correcta? (sí/parcial/no)
# - Observaciones

# ─── Resultado esperado ───
# ext3_tabla debe ser una lista de 5 dicts:
# [{"pregunta": str, "modelo_1": str, "resultado_1": str, "acierto_1": str,
#                       "modelo_2": str, "resultado_2": str, "acierto_2": str}, ...]
# acierto_X ∈ {"sí", "parcial", "no"}

ext3_tabla = []  # ← lista de 5 dicts

In [8]:
# ─── Auto-verificación ───
assert isinstance(ext3_tabla, list), "❌ ext3_tabla debe ser una lista"
assert len(ext3_tabla) == 5, f"❌ 5 preguntas esperadas, tienes {len(ext3_tabla)}"
for i, row in enumerate(ext3_tabla):
    for k in ("pregunta", "modelo_1", "resultado_1", "acierto_1", "modelo_2", "resultado_2", "acierto_2"):
        assert k in row, f"❌ Falta '{k}' en fila {i}"
    for a in ("acierto_1", "acierto_2"):
        assert row[a] in ("sí", "parcial", "no"), f"❌ {a} inválido en fila {i}"
print("✅ Ejercicio 3: formato correcto")

------------------------------------------------------------------------

## Ejercicio 4 — Agente supervisor con 2 sub-agentes (2.5 puntos)

Implementa una arquitectura **Supervisor** usando LangGraph: un agente
principal (supervisor) que decide si delegar al **Agente Analista**
(consulta_pandas, correlacion) o al **Agente Visualizador** (histograma,
barras, boxplot, resumen_estadistico).

Prueba con 3 preguntas: una que requiera solo análisis, una solo
visualización, y una que requiera ambos.

| Criterio                                      | Puntos |
|-----------------------------------------------|--------|
| 2 sub-agentes especializados implementados    | 1.0    |
| Supervisor funcional que delega correctamente | 1.0    |
| 3 preguntas evaluadas en `ext4_log`           | 0.5    |

In [9]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END

# ─── Define los sub-agentes ───
# Sub-agente 1: Analista → herramientas: consulta_pandas, correlacion
# Sub-agente 2: Visualizador → herramientas: histograma, barras, boxplot

# ─── Define el estado compartido ───
class EstadoSupervisor(TypedDict):
    pregunta: str
    tipo_tarea: str          # "analisis", "visualizacion", "ambas"
    resultado_analisis: str
    resultado_visualizacion: str
    respuesta_final: str

# ─── Nodo Supervisor ───
def nodo_supervisor(state: EstadoSupervisor) -> EstadoSupervisor:
    """El supervisor decide qué sub-agente(s) necesita."""
    # Usa el LLM para clasificar la pregunta
    # TODO: implementar lógica de clasificación
    return state

# ─── Nodo Analista ───
def nodo_analista(state: EstadoSupervisor) -> EstadoSupervisor:
    """Ejecuta el sub-agente analista."""
    # TODO: invocar sub-agente con herramientas de análisis
    return state

# ─── Nodo Visualizador ───
def nodo_visualizador(state: EstadoSupervisor) -> EstadoSupervisor:
    """Ejecuta el sub-agente visualizador."""
    # TODO: invocar sub-agente con herramientas de visualización
    return state

# ─── Construye el grafo ───
# grafo = StateGraph(EstadoSupervisor)
# ...

# ─── Preguntas de prueba ───
# 1. Solo análisis: "¿Cuál es la correlación entre edad y tarifa?"
# 2. Solo visualización: "Genera un histograma de edades"
# 3. Ambas: "Analiza la supervivencia por clase y genera un gráfico"

# ─── Resultado esperado ───
# ext4_log debe ser una lista de 3 dicts:
# [{"pregunta": str, "tipo_detectado": str, "subagentes_usados": list[str], "respuesta": str}, ...]

ext4_log = []  # ← lista de 3 dicts

In [10]:
# ─── Auto-verificación ───
assert isinstance(ext4_log, list), "❌ ext4_log debe ser una lista"
assert len(ext4_log) == 3, f"❌ 3 preguntas esperadas, tienes {len(ext4_log)}"
for i, row in enumerate(ext4_log):
    for k in ("pregunta", "tipo_detectado", "subagentes_usados", "respuesta"):
        assert k in row, f"❌ Falta '{k}' en fila {i}"
    assert isinstance(row["subagentes_usados"], list), "❌ subagentes_usados debe ser lista"
    assert len(row["subagentes_usados"]) >= 1, "❌ Al menos 1 sub-agente debe usarse"
print("✅ Ejercicio 4: formato correcto")